# Chinook Music Store Exploration  
**Dataset:** [Chinook Dataset (Github)](https://github.com/lerocha/chinook-database)   
*Exploratory Data Analysis*

In [ ]:
## re
import os
from dotenv import load_dotenv

load_dotenv()
%load_ext sql
%sql $DATABASE_URL

%config SqlMagic.feedback = False
%config SqlMagic.displaylimit = None

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


displaylimit: Value None will be treated as 0 (no limit)

**DERIVING THE MINUTES COLUMN FROM THE MILLISECONDS COLUMN**

In [4]:
%%sql
ALTER TABLE track
ADD COLUMN IF NOT EXISTS 
  minutes NUMERIC(10,2);

++
||
++
++

In [5]:
%%sql
UPDATE track
  SET minutes = ROUND(milliseconds/60000.0,2);

++
||
++
++

In [10]:
%%sql
SELECT *
FROM playlist_track
LIMIT 5;

playlist_id,track_id
1,3402
1,3389
1,3390
1,3391
1,3392


**DELETING DUPLICATE RECORDS FROM THE TRACK TABLE**

Deleting both confirmed duplicates, keeping the lower ID (1 and 3) as the "original," removing the duplicate copy (8 and 10)

BUT FIRST ....   
Check whether playlist_track's foreign key would block or cascade a delete

In [9]:
%%sql
SELECT tc.constraint_name,
        tc.table_name,
        kcu.column_name,
        rc.delete_rule
FROM information_schema.table_constraints tc
JOIN information_schema.key_column_usage kcu ON tc.constraint_name = kcu.constraint_name
JOIN information_schema.referential_constraints rc ON tc.constraint_name = rc.constraint_name
WHERE tc.table_name = 'playlist_track' AND tc.constraint_type = 'FOREIGN KEY'
;

constraint_name,table_name,column_name,delete_rule
playlist_track_playlist_id_fkey,playlist_track,playlist_id,NO ACTION
playlist_track_track_id_fkey,playlist_track,track_id,NO ACTION


> Delete rule = NO ACTION
> The database will refuse to delete a playlist directly if playlist_track still has rows pointing to it

CORRECT deletion order
* Remove the child rows(playlist_track) first
* The delete the parent row(playlist)

In [12]:
%%sql
-- # Deletions are wrapped in a transaction (BEGIN/COMMIT) so the result
-- # can be checked with SELECT before it's made permanent. If anything
-- # looks wrong, ROLLBACK undoes it completely.
BEGIN;
-- remove the duplicate playlist's track entries first
-- (required, since playlist_track references playlist and won't
-- allow an orphaned row to be created)

DELETE FROM playlist_track WHERE playlist_id = 8;
DELETE FROM playlist_track WHERE playlist_id = 10;


++
||
++
++

In [17]:
%%sql
SELECT playlist_id,
       name
FROM playlist
WHERE name IN ('TV Shows', 'Movies', 'Audiobooks', 'Music')
ORDER BY name, playlist_id;

playlist_id,name
4,Audiobooks
6,Audiobooks
2,Movies
7,Movies
1,Music
3,TV Shows


In [14]:
%%sql
-- # now the playlist rows themselves can be safely removed
DELETE FROM playlist WHERE playlist_id = 8;
DELETE FROM playlist WHERE playlist_id = 10;
-- # Checking the result before finalizing
SELECT * FROM playlist;

playlist_id,name
1,Music
2,Movies
3,TV Shows
4,Audiobooks
5,90’s Music
6,Audiobooks
7,Movies
9,Music Videos
11,Brazilian Music
12,Classical


In [15]:
%%sql
-- # If this looks correct, run:
COMMIT;

++
||
++
++

In [16]:
%%sql
-- #If anything looks wrong instead, run:
ROLLBACK;

++
||
++
++